# Frozen_stock — offline research training in Google Colab

Run cells in order using a **CPU runtime**. No GPU is required by this sklearn workflow. Colab may use a newer host Python; the setup cell provisions an isolated managed Python 3.12 environment for the pinned scientific packages.

This notebook trains new experimental candidates, not the frozen forward model. It never connects to Replit, Redis, an exchange or a brokerage account. Never paste API keys here. Upload only the prepared four-file research bundle. Uploading sends this historical dataset and model artifact to your Google runtime.

The default input pin is our verified baseline candidate. Hashes verify artifact integrity, not the truth of provider provenance. Existing historical periods have already been examined; these are not new untouched results.

Colab can terminate sessions; download completed exports or enable the optional Drive backup. Do not use Colab for the 84-day worker. [Colab FAQ](https://research.google.com/colaboratory/faq.html)


In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, sys, tempfile, zipfile, shutil

REPO_URL = "https://github.com/vdewatha/Frozen_stock.git"
REPO_COMMIT = "1015c9bd2959f14fc0b7fb7bc9916ce22dfd646d"
EXPECTED_MANIFEST_SHA256 = "3b33ea952b92c0eaace7267cdb7d3164060c07f6d2615eb847760dd4080dc6d7"
FAMILY = "random_forest"  # Or "auto": development-only family selection.
PROFILES = ["baseline", "stress"]
SAVE_TO_DRIVE = False  # Optional: authorizes Drive access when you run its cell.

assert re.fullmatch(r"[0-9a-f]{40}", REPO_COMMIT)
assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_MANIFEST_SHA256)
assert FAMILY in ("random_forest", "auto")
assert PROFILES and len(set(PROFILES)) == len(PROFILES) and set(PROFILES) <= {"baseline", "stress"}
WORK = Path(tempfile.mkdtemp(prefix="frozen-research-", dir="/content"))
REPO = WORK / "repo"
subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
VENV = WORK / "venv"
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["venv", "--python", "3.12", "--python-preference", "only-managed", str(VENV)], check=True)
PYTHON = str(VENV / "bin/python")
subprocess.run(UV + ["pip", "install", "--python", PYTHON, "-r", str(REPO / "backend/requirements.txt")], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(REPO / "backend"), OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", ALLOW_LIVE_TRADING="false")
subprocess.run([PYTHON, "-c", "import sys,numpy,pandas,sklearn; assert sys.version_info[:2]==(3,12); assert (numpy.__version__,pandas.__version__,sklearn.__version__)==('2.0.1','2.2.2','1.5.1'); print('Pinned Python 3.12 research environment ready')"], check=True, env=ENV)
print("Pinned commit:", REPO_COMMIT)


## Upload and verify the input

Use the private `colab-input.zip` prepared locally. Do **not** upload a whole project, databases, `.env`, or provider credentials. If using a different candidate bundle, set its independently verified manifest SHA-256 above before running this cell.

ZIP extraction is restricted to the four expected regular files and bounded sizes. Model loading verifies the manifest, dataset and model hashes before training.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one prepared ZIP."
archive_name, archive_bytes = next(iter(uploaded.items()))
assert len(archive_bytes) <= 80 * 1024 * 1024, "Archive too large"
import io, stat
ALLOWED = {"manifest.json", "dataset.csv", "calibrated_model.json", "final_test_predictions.csv"}
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    entries = z.infolist()
    assert len(entries) == 4 and {i.filename for i in entries} == ALLOWED, "Unexpected/duplicate archive paths"
    assert sum(i.file_size for i in entries) <= 80 * 1024 * 1024, "Expanded archive too large"
    for item in entries:
        limit = 2*1024*1024 if item.filename == "manifest.json" else 64*1024*1024
        assert item.file_size <= limit and not item.is_dir()
        assert not stat.S_ISLNK(item.external_attr >> 16), "Symlink entry rejected"
    payload = z.read("manifest.json")
    assert hashlib.sha256(payload).hexdigest() == EXPECTED_MANIFEST_SHA256, "Wrong manifest pin"
    manifest = json.loads(payload)
    assert re.fullmatch(r"[0-9a-f]{64}", manifest["run_id"])
    SOURCE = WORK / "source" / manifest["run_id"]
    SOURCE.mkdir(parents=True, exist_ok=False)
    for name in ALLOWED:
        content = z.read(name)
        if name != "manifest.json":
            assert hashlib.sha256(content).hexdigest() == manifest["files"][name], "Artifact hash mismatch"
        (SOURCE / name).write_bytes(content)
del uploaded, archive_bytes
print("Uploaded artifact hashes verified. Strict model/data validation runs before fitting.")


## Optional Google Drive backup

Leave `SAVE_TO_DRIVE=False` to download ZIPs without granting Drive access. If enabled, this cell asks you to authorize access. Training runs on the Colab VM; only completed export ZIPs and their checksums are copied to a new Drive folder. Nothing existing is overwritten. Notebook code can access mounted Drive files; review it before authorizing.


In [ ]:
BACKUP = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP = Path("/content/drive/MyDrive/Frozen_stock_research") / WORK.name
    BACKUP.mkdir(parents=True, exist_ok=False)
print("Backup enabled" if BACKUP else "Download each completed export before disconnecting.")


## Auditable training worker

The following worker is included in the notebook so this notebook runs against the exact repository commit above without requiring unpublished helper code. It uses the repository's trainers, strict artifact validator and diagnostics. It preserves input dataset bytes, uses the declared gap policy without filling missing candles, and exports inert numeric model JSON, not pickle.

Baseline costs: 0.8% fees + 0.1% slippage per side. Stress: 1% + 0.2%. These are simulation assumptions. Changing these experiments does not authorize changing the frozen study.


In [ ]:
WORKER_SOURCE = "\"\"\"Offline Colab-compatible research job; never registers models or accesses brokers.\"\"\"\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport sys\nimport zipfile\n\ndef run(source, pin, output, profile, family):\n    import pandas as pd\n    from app.services.comparison_model import _read, _decode, load_comparison_model\n    from app.services.research_training_v3 import train_research_v3\n    from app.services.model_diagnostics import build_report, publish_report, render_markdown\n    source, output = Path(source), Path(output)\n    if output.resolve().is_relative_to(source.resolve()):\n        raise ValueError(\"Output must be outside the source artifact\")\n    costs = {\"baseline\": (.008, .001), \"stress\": (.01, .002)}\n    if profile not in costs or family not in (\"random_forest\", \"auto\"):\n        raise ValueError(\"Unknown fixed job policy\")\n    declared = _decode(_read(source / \"manifest.json\", 2*1024*1024))\n    manifest, _ = load_comparison_model(source, manifest_sha256=pin,\n        source_claim=declared[\"source_claim\"], fee_rate=declared[\"fee_rate_per_side\"],\n        slippage_rate=declared[\"slippage_rate_per_side\"], expected_horizon=declared[\"horizon_bars\"])\n    snapshot = _read(source / \"dataset.csv\", 64*1024*1024)\n    if hashlib.sha256(snapshot).hexdigest() != manifest[\"dataset_sha256\"]:\n        raise ValueError(\"Dataset changed after validation\")\n    if output.exists():\n        raise FileExistsError(\"Use a fresh job directory; completed evidence is immutable\")\n    output.mkdir(parents=True)\n    fee, slip = costs[profile]\n    candidate = train_research_v3(pd.read_csv(io.BytesIO(snapshot)), output / \"candidates\",\n        source=manifest[\"source_claim\"], horizon=manifest[\"horizon_bars\"], fee_rate=fee,\n        slippage_rate=slip, seed=manifest[\"seed\"], gap_policy=manifest.get(\"gap_policy\", \"strict\"),\n        fixed_model=None if family == \"auto\" else family, source_snapshot=snapshot)\n    model = output / \"candidates\" / candidate[\"run_id\"]\n    model_pin = hashlib.sha256((model / \"manifest.json\").read_bytes()).hexdigest()\n    report = build_report(model, manifest_sha256=model_pin)\n    report_dir = publish_report(report, output / \"diagnostics\", render_markdown(report))\n    job = dict(profile=profile, family=family, source_manifest_sha256=pin,\n        candidate_run_id=candidate[\"run_id\"], candidate_manifest_sha256=model_pin,\n        diagnostic_report_id=report[\"report_id\"], eligible_for_trading=False,\n        live_authorized=False, note=\"Previously examined history; not new forward evidence\")\n    (output / \"job.json\").write_text(json.dumps(job, indent=2))\n    files = [model / n for n in (\"manifest.json\", \"dataset.csv\", \"calibrated_model.json\", \"final_test_predictions.csv\")]\n    files += [report_dir / n for n in (\"report.json\", \"report.md\", \"files.json\")]\n    files += [output / \"job.json\"]\n    hashes = {str(p.relative_to(output)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}\n    (output / \"export-hashes.json\").write_text(json.dumps(hashes, indent=2, sort_keys=True))\n    archive = output / \"research-export.zip\"\n    with zipfile.ZipFile(archive, \"x\", zipfile.ZIP_DEFLATED) as bundle:\n        for path in files + [output / \"export-hashes.json\"]:\n            bundle.write(path, str(path.relative_to(output)))\n    return job | {\"archive\": str(archive), \"archive_sha256\": hashlib.sha256(archive.read_bytes()).hexdigest()}\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--source\", type=Path, required=True)\n    parser.add_argument(\"--manifest-sha256\", required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--profile\", choices=(\"baseline\", \"stress\"), required=True)\n    parser.add_argument(\"--family\", choices=(\"random_forest\", \"auto\"), default=\"random_forest\")\n    args = parser.parse_args()\n    print(json.dumps(run(args.source, args.manifest_sha256, args.output, args.profile, args.family)))\n"
WORKER = WORK / "run_colab_research.py"
WORKER.write_text(WORKER_SOURCE)
print("Worker SHA-256:", hashlib.sha256(WORKER_SOURCE.encode()).hexdigest())

In [ ]:
results = []
for profile in PROFILES:
    output = WORK / ("job-" + profile)
    command = [PYTHON, str(WORKER), "--source", str(SOURCE),
        "--manifest-sha256", EXPECTED_MANIFEST_SHA256, "--output", str(output),
        "--profile", profile, "--family", FAMILY]
    print("Training and diagnosing:", profile, flush=True)
    completed = subprocess.run(command, env=ENV, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stderr[-12000:])
        raise RuntimeError('Research job failed; no completed export was published for this profile.')
    result = json.loads(completed.stdout)
    export = Path(result["archive"])
    assert hashlib.sha256(export.read_bytes()).hexdigest() == result["archive_sha256"]
    if BACKUP:
        destination = BACKUP / (profile + "-" + export.name)
        with destination.open("xb") as target, export.open("rb") as source:
            shutil.copyfileobj(source, target)
        assert hashlib.sha256(destination.read_bytes()).hexdigest() == result["archive_sha256"]
        (BACKUP / (profile + "-checksum.txt")).write_text(result["archive_sha256"] + "\n")
    results.append(result)
    print(json.dumps(result, indent=2))
    files.download(str(export))
print("Complete. No model was activated or registered.")


## Read results and next steps

Each ZIP contains the numeric model, original dataset, hash-verified manifests (integrity checks, not signatures or authentication), JSON/Markdown diagnostics, job metadata and export checksums. Keep them private. Inspect final-period and development walk-forward metrics against constant baselines; hypothetical replay profit is not brokerage P&L.

A failed/incomplete job is not resumable mid-fit. Completed exports remain available; to retry, run setup again for a fresh workspace and select only the unfinished profile. Copy from Drive/downloads before the VM expires. Do not edit output files to force a rerun.

To use a candidate later, securely transfer the ZIP and independently retained checksum, validate every artifact, review leakage/costs/performance, then create a **new** prospectively frozen paper experiment. Do not replace current models or reuse examined outcomes as untouched evidence.


In [ ]:
from IPython.display import Markdown, display
for result in results:
    report_path = WORK / ("job-" + result["profile"]) / "diagnostics" / result["diagnostic_report_id"] / "report.md"
    display(Markdown(report_path.read_text()))
